In [21]:
def ChuyenThanhDangKyHieu(menh_de):
    menh_de = menh_de.replace("->", "→").replace("AND", "^").replace("OR", "v").replace("NOT", "¬")
    menh_de = menh_de.replace(" ", "")
    return menh_de


In [22]:
def PhanTich(menh_de):
    # Bien chuoi thanh cay: str (bien) | ("¬", x) | ("^"/"v"/"→", trai, phai)
    vt = [0]
    def nhin():
        return menh_de[vt[0]] if vt[0] < len(menh_de) else ""

    def keo_theo():
        trai = hoac()
        if nhin() == "→":
            vt[0] += 1
            return ("→", trai, keo_theo())
        return trai

    def hoac():
        trai = va()
        while nhin() == "v":
            vt[0] += 1
            trai = ("v", trai, va())
        return trai

    def va():
        trai = phu_dinh()
        while nhin() == "^":
            vt[0] += 1
            trai = ("^", trai, phu_dinh())
        return trai

    def phu_dinh():
        if nhin() == "¬":
            vt[0] += 1
            return ("¬", phu_dinh())
        if nhin() == "(":
            vt[0] += 1
            kq = keo_theo()
            vt[0] += 1  # bo dau ")"
            return kq
        bien = nhin()
        vt[0] += 1
        return bien

    return keo_theo()


In [23]:
def BoKeoTheo(c):
    if isinstance(c, str):
        return c
    if c[0] == "¬":
        return ("¬", BoKeoTheo(c[1]))
    trai, phai = BoKeoTheo(c[1]), BoKeoTheo(c[2])
    if c[0] == "→":
        return ("v", ("¬", trai), phai)   # A->B = ¬A v B
    return (c[0], trai, phai)


In [24]:
def LuatDeMorgan(c, phu_dinh=False):
    # Day dau ¬ vao sat bien (xu ly luon ¬¬A = A)
    if isinstance(c, str):
        return ("¬", c) if phu_dinh else c
    if c[0] == "¬":
        return LuatDeMorgan(c[1], not phu_dinh)
    trai = LuatDeMorgan(c[1], phu_dinh)
    phai = LuatDeMorgan(c[2], phu_dinh)
    if phu_dinh:
        return ("v" if c[0] == "^" else "^", trai, phai)
    return (c[0], trai, phai)


In [25]:
def LuatPhanPhoi(c):
    # Tra ve danh sach menh de (moi menh de la 1 frozenset cac literal)
    if isinstance(c, str):
        return [frozenset({c})]
    if c[0] == "¬":
        return [frozenset({"¬" + c[1]})]
    A = LuatPhanPhoi(c[1])
    B = LuatPhanPhoi(c[2])
    if c[0] == "^":
        return A + B
    return [x | y for x in A for y in B]   # v phan phoi len ^


In [26]:
def Doi(literal):
    return literal[1:] if literal[0] == "¬" else "¬" + literal

def LaHangDung(menh_de):
    return any(Doi(l) in menh_de for l in menh_de)

def InMenhDe(menh_de):
    if not menh_de:
        return "□"
    return " v ".join(sorted(menh_de, key=lambda x: x.replace("¬", "")))


In [27]:
def ChuyenThanhCNF(menh_de):
    menh_de = ChuyenThanhDangKyHieu(menh_de)
    cay = PhanTich(menh_de)
    cay = BoKeoTheo(cay)
    cay = LuatDeMorgan(cay)
    ds = []
    for m in LuatPhanPhoi(cay):
        if not LaHangDung(m) and m not in ds:
            ds.append(m)
    return ds


In [28]:
def PhanGiai(m1, m2):
    # Tim cac cap literal doi nhau, tra ve danh sach menh de giai thuc
    ket_qua = []
    for l in m1:
        if Doi(l) in m2:
            moi = (m1 - {l}) | (m2 - {Doi(l)})
            if not LaHangDung(moi):
                ket_qua.append(moi)
    return ket_qua


In [29]:
def Robinson(ds):
    i = 0
    while i < len(ds):
        for j in range(i):
            for moi in PhanGiai(ds[j], ds[i]):
                if moi not in ds:
                    ds.append(moi)
                    print(f"({len(ds)}) {InMenhDe(moi)}   [giai ({j+1}), ({i+1})]")
                    if not moi:
                        return True
        i += 1
    return False


In [30]:
# gt_input = "(a^b)->c,(b^c)->d,NOTd"   # vi du cu cua ban
# kl_input = "a->b"
gt_input = "a->b,b->c"
kl_input = "a->c"

# B1: dua gia thiet ve CNF
ds = []
for gt in gt_input.split(","):
    for m in ChuyenThanhCNF(gt):
        if m not in ds:
            ds.append(m)

# B2: phu dinh ket luan roi dua ve CNF
kl_phu_dinh = "¬(" + ChuyenThanhDangKyHieu(kl_input) + ")"
for m in ChuyenThanhCNF(kl_phu_dinh):
    if m not in ds:
        ds.append(m)

print("B1+B2: Tap menh de (GT va phu dinh KL)")
for k, m in enumerate(ds):
    print(f"({k+1}) {InMenhDe(m)}")

print("\nB3: Ap dung quy tac giai")
if Robinson(ds):
    print("\n=> Ra menh de rong (□): ket luan DUNG")
else:
    print("\n=> Khong sinh them duoc menh de nao: ket luan KHONG suy ra duoc")

B1+B2: Tap menh de (GT va phu dinh KL)
(1) ¬a v b
(2) ¬b v c
(3) a
(4) ¬c

B3: Ap dung quy tac giai
(5) ¬a v c   [giai (1), (2)]
(6) b   [giai (1), (3)]
(7) ¬b   [giai (2), (4)]
(8) c   [giai (3), (5)]
(9) ¬a   [giai (4), (5)]
(10) □   [giai (6), (7)]

=> Ra menh de rong (□): ket luan DUNG
